# Class 30: Monitoring & Observability

প্রতিটা query-র token, cost, latency আর quality score track করা। এই notebook offline চলে: FakeBackend + tempfile SQLite, Ollama/Docker/Langfuse লাগে না।

## 1. Setup: Class 2 Project import করা

In [ ]:
import sys, tempfile, os
sys.path.insert(0, os.path.abspath("../Class 2 Project"))
from app.backend import FakeBackend
from app.service import RagService, CallRecord
from app.monitor import Monitor
tmp = tempfile.mkdtemp()
mon = Monitor(os.path.join(tmp, "m.db"))
backend = FakeBackend(); backend.ingest("hr.pdf", b"%PDF-1")
svc = RagService(backend); svc.observers.append(mon.observer)   # the seam
print("observers:", len(svc.observers))

## 2. Seam দিয়ে query চালানো
`_run()` প্রতিটা call-এর শেষে `CallRecord` দেয়; `Monitor.observer` token, cost, quality হিসাব করে SQLite-এ লেখে।

In [ ]:
import asyncio
async def go():
    for q in ["leave policy days", "notice period rules", "bonus eligibility"]:
        await svc.ask(q, None, None, request_id=q[:5])
await go()   # Jupyter-এ top-level await চলে
for r in mon.rows(): print(r["question"], r["prompt_tokens"], r["completion_tokens"], r["cost_usd"], r["quality"])

## 3. Cost: illustrative price table
Local model free, তাই equivalent cloud cost দেখাই। দামগুলো শেখার জন্য গোল সংখ্যা, আসল দাম নয়।

| Key | Use case | Advantage | Limitation |
| --- | --- | --- | --- |
| small-cloud (chosen) | সস্তা RAG তুলনা | কম খরচ | reasoning মাঝারি |
| large-cloud | কঠিন reasoning | সর্বোচ্চ quality | অনেক দামি |
| local | আসল চলমান model | খরচ ০ | hardware লাগে |

In [ ]:
from app.pricing import DEFAULT_TABLE, equivalent_cost
print(DEFAULT_TABLE)
for k in DEFAULT_TABLE: print(k, equivalent_cost(k, 2000, 500))

## 4. Quality score: grounding + refusal

In [ ]:
from app.quality import grounding_score
ctx = ["Employees get twenty days of annual leave per year."]
print(grounding_score("Employees get twenty days of annual leave", ctx, True))
print(grounding_score("The company offers free bitcoin and rocket trips", ctx, True))
print(grounding_score("This information was not found in the documents.", [], False))

## 5. Drift: baseline বনাম recent window
প্রথম N query baseline, শেষ N recent। Latency বাড়লে বা quality কমলে alert।

In [ ]:
from app.quality import drift_report
rows = [{"latency_s": 0.2, "quality": 0.9}] * 20 + [{"latency_s": 0.5, "quality": 0.6}] * 20
print(drift_report(rows, 20))

## 6. Stats (যা `/stats` endpoint দেয়)

In [ ]:
import json
s = mon.stats(drift_n=20)
print(json.dumps({k: v for k, v in s.items() if k != "recent"}, indent=2, default=str))

## 7. Langfuse sink: env না থাকলে বন্ধ
Observability tool-এর তুলনা project-এর `app/langfuse_obs.py` HIGHLIGHTS-এ আছে (Langfuse vs LangSmith vs Phoenix vs Helicone)।

In [ ]:
from app.langfuse_obs import make_langfuse_sink
print("sink without env:", make_langfuse_sink())   # None -> app স্বাভাবিক চলে

## 8. Docker Compose primer
Langfuse self-host করতে (project folder-এ): `cp .env.example .env` → `docker compose config` (যাচাই) → `docker compose up -d` → http://localhost:3000।
Service: langfuse-web, langfuse-worker, postgres, clickhouse, redis, minio। `down -v` volume মুছে দেয়।

In [ ]:
import subprocess
r = subprocess.run(["docker", "compose", "config", "-q"], cwd="../Class 2 Project", capture_output=True, text=True)
print("compose valid" if r.returncode == 0 else r.stderr[:300])

## Exercise
1. `backend.answer_override` বদলে hallucinated উত্তর বানান; quality কত নামে?
2. `RAG_PRICE_TABLE_JSON` দিয়ে নিজের price key যোগ করুন।
3. Drift threshold বদলে alert কখন আসে দেখুন।